In [1]:
import math
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

def bucketize_dataset_for_cartesian(
    df: pd.DataFrame,
    candidate_cols: list[str],
    target_cartesian_size: int = 700,
    max_bins_per_col: int = 16,
    min_bins_per_col: int = 1,
    drop_id_like: bool = True,
    id_like_threshold: float = 0.90,
    other_label: str = "OTHER",
    all_label: str = "ALL",
    verbose: bool = True,
):
    """
    Bucketize selected columns so that the PRODUCT of their unique counts
    (i.e., the size of the Cartesian domain) is <= target_cartesian_size.

    Strategy
    --------
    1) Filter out ID-like columns (>= id_like_threshold * n_rows unique).
    2) Choose per-column target buckets K_j with a simple greedy shrink loop:
         - Start K_j = min(nunique_j, max_bins_per_col)  (>= min_bins_per_col)
         - While ∏ K_j > target: decrement the column with the largest K_j (but >= min_bins_per_col)
    3) Apply bucketization:
         - Numeric: quantile bins (pd.qcut with duplicates='drop'); labeled as "[lo, hi]"
         - Categorical:
             * If nunique <= K_j: keep categories as-is
             * Else: keep top (K_j-1) categories and map the rest to OTHER
           If K_j == 1, map everything to ALL
    4) Return the bucketized df (columns replaced in-place), plus metadata:
         used_cols, ignored_cols, per_col_bins (target & realized), final_product, etc.

    Notes
    -----
    - If qcut collapses adjacent quantiles (due to repeated values), the realized
      number of bins can be < K_j — that only *reduces* the final product further.
    - If a column is binary already (nunique <= 2), it won't be expanded.
    - This function modifies only the columns in `candidate_cols` and returns
      a shallow copy of the original df with bucketized values in those columns.

    Returns
    -------
    df_bkt : pd.DataFrame
        Copy of df with selected columns bucketized (overwritten).
    info : dict
        {
          'target_cartesian_size': int,
          'used_cols': list[str],
          'ignored_cols': list[str],
          'reason_ignored': dict[col -> str],
          'target_bins': dict[col -> int],
          'realized_bins': dict[col -> int],
          'final_product': int,
          'per_col_uniques': dict[col -> int],
        }
    """

    if target_cartesian_size < 1:
        raise ValueError("target_cartesian_size must be >= 1")

    n_rows = len(df)
    df_bkt = df.copy()

    # --- 1) Filter candidate columns (drop ID-like if requested) ---
    ignored = []
    reason_ignored = {}
    used = []

    # Precompute nunique
    nunique = {c: int(df[c].nunique(dropna=False)) for c in candidate_cols}

    for c in candidate_cols:
        if drop_id_like and n_rows > 0 and nunique[c] >= id_like_threshold * n_rows:
            ignored.append(c)
            reason_ignored[c] = f"ID-like: nunique={nunique[c]} (~{nunique[c]/max(1,n_rows):.1%} of rows)"
        else:
            used.append(c)

    if not used:
        info = {
            "target_cartesian_size": target_cartesian_size,
            "used_cols": [],
            "ignored_cols": ignored,
            "reason_ignored": reason_ignored,
            "target_bins": {},
            "realized_bins": {},
            "final_product": 0,
            "per_col_uniques": {},
        }
        return df_bkt, info

    # --- 2) Decide target bins K_j per used column (greedy shrink) ---
    K = {}
    for c in used:
        Kj = max(min_bins_per_col, min(nunique[c], max_bins_per_col))
        K[c] = Kj

    def product_K(_K: dict) -> int:
        prod = 1
        for v in _K.values():
            prod *= max(1, int(v))
        return prod

    # Greedily shrink the largest K_j until product <= target
    # (simple, deterministic, and effective in practice)
    while product_K(K) > target_cartesian_size:
        # Pick col with largest K (tie: largest nunique; then column name)
        c = max(K.keys(), key=lambda col: (K[col], nunique[col], col))
        if K[c] > min_bins_per_col:
            K[c] -= 1
        else:
            # All columns are at min already → can't shrink further
            break

    target_bins = K.copy()

    # --- 3) Apply bucketization per column ---
    realized_bins = {}
    per_col_uniques = {}

    for c in used:
        s = df_bkt[c]
        Kj = int(target_bins[c])
        if Kj <= 1:
            df_bkt[c] = all_label
            realized_bins[c] = 1
            per_col_uniques[c] = 1
            continue

        if is_numeric_dtype(s):
            # Numeric → quantile bins into Kj buckets, allow duplicates='drop'
            try:
                cats = pd.qcut(s, q=Kj, duplicates="drop")
                # Replace with interval labels for nice distinct strings
                labels = cats.cat.categories
                # Map category codes to "[lo, hi]" strings
                lab_map = {cat: f"[{cat.left},{cat.right}]" for cat in labels}
                df_bkt[c] = cats.map(lab_map).astype("category")
                realized_bins[c] = len(labels)
            except Exception:
                # Fallback: cut into equal-width bins; if still fails → ALL
                try:
                    cats = pd.cut(s, bins=min(Kj, max(1, s.nunique())), include_lowest=True)
                    labels = cats.cat.categories
                    lab_map = {cat: f"[{cat.left},{cat.right}]" for cat in labels}
                    df_bkt[c] = cats.map(lab_map).astype("category")
                    realized_bins[c] = len(labels)
                except Exception:
                    df_bkt[c] = all_label
                    realized_bins[c] = 1
        else:
            # Categorical/string
            vals = s.astype("object")
            # How many distinct do we want to KEEP explicitly?
            # If nunique <= Kj: keep all categories as-is.
            # Else: keep top (Kj-1), map the rest to OTHER to reach Kj total.
            u = int(vals.nunique(dropna=False))
            if u <= Kj:
                df_bkt[c] = vals.astype("category")
                realized_bins[c] = u
            else:
                # Keep top Kj-1
                top = vals.value_counts(dropna=False).sort_values(ascending=False)
                keep = set(top.iloc[:Kj-1].index.tolist())
                df_bkt[c] = vals.where(vals.isin(keep), other_label).astype("category")
                realized_bins[c] = int(df_bkt[c].nunique(dropna=False))

        per_col_uniques[c] = int(df_bkt[c].nunique(dropna=False))

    # Recompute final product on realized uniques (<= target by construction)
    final_product = 1
    for c in used:
        final_product *= max(1, int(per_col_uniques[c]))

    info = {
        "target_cartesian_size": target_cartesian_size,
        "used_cols": used,
        "ignored_cols": ignored,
        "reason_ignored": reason_ignored,
        "target_bins": target_bins,
        "realized_bins": realized_bins,
        "final_product": final_product,
        "per_col_uniques": per_col_uniques,
    }

    if verbose:
        print("[Bucketize] used:", used)
        if ignored:
            print("[Bucketize] ignored:", {c: reason_ignored[c] for c in ignored})
        print("[Bucketize] target bins:", target_bins)
        print("[Bucketize] realized uniques:", per_col_uniques, "| final product =", final_product)

    return df_bkt, info


In [2]:

def bucketize_adult_census(df):
    """
    Applies the most sensible, knowledge-driven bucketizing strategies specifically
    for the UCI Adult Census dataset.

    Args:
        df (pd.DataFrame): The input DataFrame with Adult Census data.

    Returns:
        pd.DataFrame: A new DataFrame with transformed and added columns.
    """
    df_new = df.copy()

    # --- 🧑 Age: Use standard demographic groups ---
    age_bins = [16, 25, 40, 60, 100]
    age_labels = ['Young Adult', 'Middle-Aged', 'Senior', 'Elder']
    df_new['age_group'] = pd.cut(df_new['age'], bins=age_bins, labels=age_labels, right=True)

    # --- ⏰ hours_per_week: Use common work standards ---
    hours_bins = [0, 39, 41, 100]
    hours_labels = ['Part-Time', 'Full-Time', 'Over-Time']
    df_new['work_hours_cat'] = pd.cut(df_new['hours_per_week'], bins=hours_bins, labels=hours_labels, right=True)

    # --- 💰 capital_gain & capital_loss: Simplify skewed data ---
    df_new['capital_change'] = np.where(df_new['capital_gain'] > 0, 'Gain',
                                     np.where(df_new['capital_loss'] > 0, 'Loss', 'None'))

    # --- 🗺️ native_country: Simplify into USA vs. Non-USA ---
    df_new['country_group'] = np.where(df_new['native_country'].str.strip() == 'United-States', 'USA', 'Non-USA')

    return df_new

In [8]:
df=pd.read_csv("../data/real/pricerunner_aggregate.csv")
#unique values before
print(df.nunique())


ProductID        35311
ProductTitle     30993
MerchantID         306
ClusterID        13233
ClusterLabel     12849
CategoryID          10
CategoryLabel       10
dtype: int64


In [9]:
df_bkt.head()

,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,age_group,work_hours_cat,capital_change,country_group
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,Middle-Aged,Full-Time,Gain,USA
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,Senior,Part-Time,None,USA
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,Middle-Aged,Full-Time,None,USA
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,Senior,Full-Time,None,USA
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,Middle-Aged,Full-Time,None,Non-USA


In [10]:
#save to csv
df_bkt.to_csv("../data/real/census_bucketized.csv", index=False)

In [11]:
df_bkt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   age             48842 non-null  int64   
 1   workclass       48842 non-null  object  
 2   education       48842 non-null  object  
 3   education_num   48842 non-null  int64   
 4   marital_status  48842 non-null  object  
 5   occupation      48842 non-null  object  
 6   relationship    48842 non-null  object  
 7   race            48842 non-null  object  
 8   sex             48842 non-null  object  
 9   capital_gain    48842 non-null  int64   
 10  capital_loss    48842 non-null  int64   
 11  hours_per_week  48842 non-null  int64   
 12  native_country  48842 non-null  object  
 13  age_group       48842 non-null  category
 14  work_hours_cat  48842 non-null  category
 15  capital_change  48842 non-null  object  
 16  country_group   48842 non-null  object  
dtypes: category(

In [13]:
import pandas as pd
import numpy as np
import io

def bucketize_compas_data(df):
    """
    Applies sensible, knowledge-driven bucketizing strategies specifically
    for the ProPublica COMPAS dataset columns.

    Args:
        df (pd.DataFrame): The input DataFrame with COMPAS data.

    Returns:
        pd.DataFrame: A new DataFrame with added bucketized columns.
    """
    df_new = df.copy()

    # --- 🧑 age: Standard demographic groups ---
    # Bins: <25 (Young Adult), 25-45 (Adult), >45 (Older Adult)
    df_new['age_cat_binned'] = pd.cut(df_new['age'],
                                      bins=[0, 25, 45, np.inf],
                                      labels=['<25', '25-45', '>45'])

    # --- ⚖️ priors_count: Bins reflecting escalating criminal history ---
    # Bins: 0, 1-3, 4-10, 11+ priors
    df_new['priors_cat'] = pd.cut(df_new['priors_count'],
                                  bins=[-1, 0, 3, 10, np.inf],
                                  labels=['0 Priors', '1-3 Priors', '4-10 Priors', '11+ Priors'])

    # --- 🧒 juv_other_count: Simple bins for juvenile history ---
    df_new['juv_other_cat'] = pd.cut(df_new['juv_other_count'],
                                     bins=[-1, 0, 1, np.inf],
                                     labels=['0', '1', '2+'])

    # --- 🗓️ Temporal relationships: days_b_screening_arrest & c_days_from_compas ---
    # Bins capture if the event was long before, around the same time, or after the screening
    df_new['screening_arrest_timing'] = pd.cut(df_new['days_b_screening_arrest'],
                                               bins=[-np.inf, -30, -1, 1, 30, np.inf],
                                               labels=['>30 Days Before', '1-30 Days Before', 'Same Day', '1-30 Days After', '>30 Days After'])

    df_new['offense_compas_timing'] = pd.cut(df_new['c_days_from_compas'],
                                            bins=[-np.inf, -1, 1, np.inf],
                                            labels=['Before Screening', 'During Screening', 'After Screening'])

    # --- ⏳ Durations: start & end ---
    # Bins capture short, medium, and long durations
    df_new['start_duration'] = pd.cut(df_new['start'],
                                      bins=[-1, 0, 90, 365, np.inf],
                                      labels=['Immediate', '1-3 Months', '3-12 Months', '>1 Year'])

    df_new['end_duration'] = pd.cut(df_new['end'],
                                    bins=[-1, 90, 365, 730, np.inf],
                                    labels=['<3 Months', '3-12 Months', '1-2 Years', '>2 Years'])

    return df_new



In [14]:
#compass
df_compas = pd.read_csv("../data/real/compas.csv")
df_compas_bkt = bucketize_compas_data(df_compas)

In [15]:
df_compas_bkt.head()

,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,end,event,two_year_recid,age_cat_binned,priors_cat,juv_other_cat,screening_arrest_timing,offense_compas_timing,start_duration,end_duration
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,327,0,0,>45,0 Priors,0,1-30 Days Before,During Screening,Immediate,3-12 Months
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,159,1,1,25-45,0 Priors,0,1-30 Days Before,During Screening,1-3 Months,3-12 Months
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,63,0,1,<25,4-10 Priors,1,1-30 Days Before,During Screening,Immediate,<3 Months
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,1174,0,0,<25,1-3 Priors,0,NaN,During Screening,Immediate,>2 Years
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1102,0,0,25-45,1-3 Priors,0,NaN,After Screening,Immediate,>2 Years


In [16]:
df_compas_bkt.to_csv("../data/real/compas_bucketized.csv", index=False)

In [17]:
df_compas_bkt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7214 entries, 0 to 7213
Data columns (total 60 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   id                       7214 non-null   int64   
 1   name                     7214 non-null   object  
 2   first                    7214 non-null   object  
 3   last                     7214 non-null   object  
 4   compas_screening_date    7214 non-null   object  
 5   sex                      7214 non-null   object  
 6   dob                      7214 non-null   object  
 7   age                      7214 non-null   int64   
 8   age_cat                  7214 non-null   object  
 9   race                     7214 non-null   object  
 10  juv_fel_count            7214 non-null   int64   
 11  decile_score             7214 non-null   int64   
 12  juv_misd_count           7214 non-null   int64   
 13  juv_other_count          7214 non-null   int64   
 14  priors_c

In [18]:
#keep sampling from thes column candidate_cols=['c_days_from_compas', 'juv_other_count', 'days_b_screening_arrest', 'priors_count', 'age_cat_binned', 'start_duration','end_duration']
#until total cartesian product of unique values across these columns is <=700

candidate_cols=['c_days_from_compas', 'juv_other_count', 'days_b_screening_arrest', 'priors_count', 'age_cat_binned', 'start_duration','end_duration']
df_compas_bkt_bkt, info = bucketize_dataset_for_cartesian(
    df_compas_bkt,
    candidate_cols=candidate_cols,
    target_cartesian_size=700,
    max_bins_per_col=16,
    min_bins_per_col=1,
    drop_id_like=True,
    id_like_threshold=0.90,
    other_label="OTHER",
    all_label="ALL",
    verbose=True,
)

[Bucketize] used: ['c_days_from_compas', 'juv_other_count', 'days_b_screening_arrest', 'priors_count', 'age_cat_binned', 'start_duration', 'end_duration']
[Bucketize] target bins: {'c_days_from_compas': 2, 'juv_other_count': 3, 'days_b_screening_arrest': 2, 'priors_count': 2, 'age_cat_binned': 3, 'start_duration': 3, 'end_duration': 3}
[Bucketize] realized uniques: {'c_days_from_compas': 3, 'juv_other_count': 1, 'days_b_screening_arrest': 3, 'priors_count': 2, 'age_cat_binned': 3, 'start_duration': 3, 'end_duration': 3} | final product = 486


In [19]:
df_compas_bkt_bkt.head()

,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,end,event,two_year_recid,age_cat_binned,priors_cat,juv_other_cat,screening_arrest_timing,offense_compas_timing,start_duration,end_duration
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,327,0,0,>45,0 Priors,0,1-30 Days Before,During Screening,Immediate,3-12 Months
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,159,1,1,25-45,0 Priors,0,1-30 Days Before,During Screening,1-3 Months,3-12 Months
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,63,0,1,<25,4-10 Priors,1,1-30 Days Before,During Screening,Immediate,OTHER
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,1174,0,0,<25,1-3 Priors,0,NaN,During Screening,Immediate,>2 Years
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1102,0,0,25-45,1-3 Priors,0,NaN,After Screening,Immediate,>2 Years


In [20]:
df_compas_bkt_bkt.to_csv("../data/real/compas_bucketized_for_cartesian.csv", index=False)

In [36]:
import pandas as pd
import io

def simplify_product_columns_inplace(df, top_n_merchants=50):
    """
    Simplifies product data columns in-place to reduce cardinality,
    aiming for < 100 unique values per column without creating new columns.

    Args:
        df (pd.DataFrame): The input product DataFrame.
        top_n_merchants (int): The number of top merchants to preserve.

    Returns:
        pd.DataFrame: A new DataFrame with modified columns.
    """
    df_new = df.copy()

    # 1. Standardize 'ProductTitle' by replacing it with 'ClusterLabel'
    print("⚙️ Standardizing ProductTitle using ClusterLabel...")
    df_new['ProductTitle'] = df_new['ClusterLabel']

    # 2. Simplify 'MerchantID' by grouping rare merchants
    # Find the top N merchants
    top_merchants = df_new['MerchantID'].value_counts().nlargest(top_n_merchants).index
    # Replace any merchant ID not in the top list with a placeholder (e.g., 0 for 'Other')
    print(f"⚙️ Simplifying MerchantID to Top {top_n_merchants} + 'Other' (ID 0)...")
    df_new.loc[~df_new['MerchantID'].isin(top_merchants), 'MerchantID'] = 0

    # 3. Drop 'ProductID' as it's a unique identifier that can't be simplified
    print("🧹 Dropping unique identifier column 'ProductID'...")
    df_new = df_new.drop(columns=['ProductID'])

    return df_new

# --- Example Usage ---
csv_data = """ProductID,ProductTitle,MerchantID,ClusterID,ClusterLabel,CategoryID,CategoryLabel
1,apple iphone 8 plus 64gb silver,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,apple iphone 8 plus 64 gb spacegrau,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim free smartphone in gold,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
4,Samsung Galaxy S21 128GB Phantom Black,4,2,Samsung Galaxy S21,2612,Mobile Phones
5,SAMSUNG Galaxy S21 5G 128GB,999,2,Samsung Galaxy S21,2612,Mobile Phones
"""

df = pd.read_csv(io.StringIO(csv_data))

# Process the data using a smaller top_n for this example
processed_df = simplify_product_columns_inplace(df, top_n_merchants=2)

print("\n--- Column Cardinality Before ---")
print(df.nunique())

print("\n--- Column Cardinality After ---")
print(processed_df.nunique())

print("\n--- Processed Data ---")
display(processed_df)

⚙️ Standardizing ProductTitle using ClusterLabel...
⚙️ Simplifying MerchantID to Top 2 + 'Other' (ID 0)...
🧹 Dropping unique identifier column 'ProductID'...

--- Column Cardinality Before ---
ProductID        5
ProductTitle     5
MerchantID       4
ClusterID        2
ClusterLabel     2
CategoryID       1
CategoryLabel    1
dtype: int64

--- Column Cardinality After ---
ProductTitle     2
MerchantID       3
ClusterID        2
ClusterLabel     2
CategoryID       1
CategoryLabel    1
dtype: int64

--- Processed Data ---


,ProductTitle,MerchantID,ClusterID,ClusterLabel,CategoryID,CategoryLabel
0,Apple iPhone 8 Plus 64GB,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
1,Apple iPhone 8 Plus 64GB,2,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
2,Apple iPhone 8 Plus 64GB,1,1,Apple iPhone 8 Plus 64GB,2612,Mobile Phones
3,Samsung Galaxy S21,0,2,Samsung Galaxy S21,2612,Mobile Phones
4,Samsung Galaxy S21,0,2,Samsung Galaxy S21,2612,Mobile Phones


In [39]:
pricerunner_df = pd.read_csv("../data/real/pricerunner_aggregate.csv")
pricerunner_df_simplified = simplify_product_columns_inplace(pricerunner_df)
#get first word from cluster label and put it in a new column called brand
pricerunner_df_simplified['brand'] = pricerunner_df_simplified['ClusterLabel'].str.split().str[0]
#keep top 10 brands and replace the rest with 'Other'
top_brands = pricerunner_df_simplified['brand'].value_counts().nlargest(50).index
pricerunner_df_simplified.loc[~pricerunner_df_simplified['brand'].isin(top_brands), 'brand'] = 'Other'


⚙️ Standardizing ProductTitle using ClusterLabel...
⚙️ Simplifying MerchantID to Top 50 + 'Other' (ID 0)...
🧹 Dropping unique identifier column 'ProductID'...


In [40]:
#unique values in each column
print(pricerunner_df_simplified.nunique())

ProductTitle     12849
MerchantID          51
ClusterID        13233
ClusterLabel     12849
CategoryID          10
CategoryLabel       10
brand               51
dtype: int64


In [41]:
#save pricerunner simplified to csv
pricerunner_df_simplified.to_csv("../data/real/pricerunner_simplified.csv", index=False)

In [54]:
# Bin the 'price' column into 20 quantile-based bins
df=pd.read_csv("../data/real/flights.csv")
df['price_category'] = pd.qcut(df['price'], q=5, duplicates='drop')

# Display the result
print("Price column binned into 20 categories:")
print(df[['price', 'price_category']].head())


Price column binned into 20 categories:
   price    price_category
0   5953  (4394.0, 6133.0]
1   5953  (4394.0, 6133.0]
2   5956  (4394.0, 6133.0]
3   5955  (4394.0, 6133.0]
4   5955  (4394.0, 6133.0]


In [61]:
#save to csv
df.to_csv("../data/real/flights_bucketized.csv", index=False)

In [62]:
#unique values in each column
print(df.nunique())

Unnamed: 0           300153
airline                   6
flight                 1561
source_city               6
departure_time            6
stops                     3
arrival_time              6
destination_city          6
class                     2
duration                476
days_left                49
price                 12157
price_category            5
duration_category         5
days_category            13
dtype: int64


In [64]:
#also bin duration based on highest 20 quantiles
df['duration_category'] = pd.qcut(df['duration'], q=5, duplicates='drop')
#save to csv

#bucketize days into weeks if days>7 else 1
df['days_category'] = df['days_left'].apply(lambda x: f"{(x-1)//7 + 1} week(s)" if x > 7 else "1 week")


In [65]:
#unique values in each column
print(df.nunique())

Unnamed: 0           300153
airline                   6
flight                 1561
source_city               6
departure_time            6
stops                     3
arrival_time              6
destination_city          6
class                     2
duration                476
days_left                49
price                 12157
price_category            5
duration_category         5
days_category             7
dtype: int64


In [66]:
#census unique values
df=pd.read_csv("../data/real/census.csv")
print(df.nunique())

age                74
workclass           9
education          16
education_num      16
marital_status      7
occupation         15
relationship        6
race                5
sex                 2
capital_gain      123
capital_loss       99
hours_per_week     96
native_country     42
dtype: int64


In [67]:
#compas unique values
df=pd.read_csv("../data/real/compas.csv")
print(df.nunique())

id                         7214
name                       7158
first                      2800
last                       3950
compas_screening_date       690
sex                           2
dob                        5452
age                          65
age_cat                       3
race                          6
juv_fel_count                11
decile_score                 10
juv_misd_count               10
juv_other_count              10
priors_count                 37
days_b_screening_arrest     423
c_jail_in                  6907
c_jail_out                 6880
c_case_number              7192
c_offense_date              927
c_arrest_date               580
c_days_from_compas          499
c_charge_degree               2
c_charge_desc               437
is_recid                      2
r_case_number              3471
r_charge_degree              10
r_days_from_arrest          201
r_offense_date             1075
r_charge_desc               340
r_jail_in                   972
r_jail_o

(1426337, 11)
asin                 1426337
title                1385430
imgUrl               1372162
productURL           1426337
stars                     41
reviews                11861
price                  29961
listPrice              14518
category_id              248
isBestSeller               2
boughtInLastMonth         30
dtype: int64
